Load the TensorRT fusion model (two inputs: camera + VL53L0X).

In [ ]:
# Full reset of the sensors and camera
!echo 'jetson' | sudo -S bash scripts/sensor_soft_reset.sh && printf '\n'
!echo 'jetson' | sudo -S systemctl restart nvargus-daemon && printf '\n'

import torch
from torch2trt import TRTModule
from scripts.xy_dataset import normalize_tof_tensor

model_trt = TRTModule()
model_trt.load_state_dict(torch.load('road_following_model_sensor_trt.pth'))

Create the racecar class

In [ ]:
from robot.jetracer import JetRacer

car = JetRacer(bus=7, signal_freq=50, servo_channel=0, motor_channel=1)

Create the camera class.

In [ ]:
from jetcam.csi_camera import CSICamera

camera = CSICamera(width=224, height=224, capture_fps=15, flip_method=2)

Create the VL53L0X pair (I2C bus 1).

In [ ]:
from robot.vl53l0x import VL53Pair

tof = VL53Pair(bus=1, addr_left=0x28, addr_right=0x29)
print('ToF mm', tof.read_mm())

In [ ]:
# enable the oled display. The server listens on this Jetson, so localhost
import requests
BASE_URL = "http://127.0.0.1:8000/stats"

for action in ("off", "on"):
    response = requests.get(f"{BASE_URL}/{action}", timeout=5)
    print(f"{action.upper()}:{response.status_code}")

In [ ]:
from jetcard.widgets import Menu, FloatSlider, IntSlider, Button, reset_menu
from IPython.display import display
import time

reset_menu()
throttle = FloatSlider(min=-1.0, max=1.0, step=0.01, value=0.1, description='throttle')
str_gain = FloatSlider(min=-5.0, max=5.0, step=0.05, value=0.8, description='steering gain')
str_bias = FloatSlider(min=-1.0, max=1.0, step=0.01, value=0.0, description='steering bias')

def _reset_sensors(func_obj, button):
    tof.request_reset(blocking=True)

reset_sensors = Button(description='reset sensors')
reset_sensors.on_click(_reset_sensors)

In [ ]:
from scripts.utils import preprocess
import numpy as np

# ===== throttle control =====
car.throttle = throttle.value = 0.1

while True:
    
    st = time.time()
    image = camera.read()
    image = preprocess(image).half()

    tof_left, tof_right = tof.read_mm()  # (2000, 2000) while a read is failing; repeated failures reset the sensors
    sensors = normalize_tof_tensor(
        torch.tensor([[tof_left, tof_right]], dtype=torch.float32, device='cuda')
    ).half()

    output = model_trt(image, sensors).detach().cpu().numpy().flatten()
    x = float(output[0])

    # ===== steering control =====
    car.steering = x * str_gain.value + str_bias.value

    # print(car.steering, car.throttle)
    # print(time.time()-st)